In [1]:
import random
import math
import statistics
import matplotlib.pyplot as plt
import numpy as np

def info_market_maker(Information_fractions,seed,volatility=.2,trade_steps=1000,information_steps=100,customer_uncertainty=.8,spread = .2, informed_trade_probability=.7):
    rng = random.Random(seed)
    InformationProfit = []
    AvgTrades = []
    InformationLossRates = []
    PnlStdevs = []
    for informed_fraction in Information_fractions:
        total_profit = 0.0
        total_trades = 0
        losses = 0
        run_pnls = []
        for i in range(information_steps):
            fair_price = 100.0
            inventory = 0
            cash = 0.0
            max_inventory = 10
            order_flow_bias = .4
            arrival_sensitivity = .7
            trades = 0
            pnl = 0
            transaction_cost = .02
            for t in range(trade_steps):
                price_change = rng.gauss(0, volatility)

                bid = fair_price - spread / 2
                ask = fair_price + spread / 2

                arrival_prob = math.exp(-arrival_sensitivity * spread)
                s = rng.random()
                if s < arrival_prob:
                    if s < informed_fraction:
                        if price_change > 0:
                            if s < informed_trade_probability:
                                buy_probability = (.5 + order_flow_bias)
                            else:
                                buy_probability = (.5 - order_flow_bias)
                        elif price_change <0:
                            if s < informed_trade_probability:
                                buy_probability = (.5 - order_flow_bias)
                            else:
                                buy_probability = (.5 + order_flow_bias)
                        else:
                            buy_probability = .5
                        if s < buy_probability and inventory > -max_inventory:
                            inventory -= 1
                            cash += ask - transaction_cost
                            trades += 1
                        elif s >= buy_probability and inventory < max_inventory:
                            inventory += 1
                            cash -= bid + transaction_cost
                            trades += 1
                    else:
                        customer_min_surplus = .05
                        customer_value = fair_price + rng.gauss(0, customer_uncertainty)
                        buy_surplus = customer_value - ask
                        sell_surplus = bid - customer_value
                        if buy_surplus > customer_min_surplus and buy_surplus >= sell_surplus and inventory > -max_inventory:
                            inventory -= 1
                            cash += ask - transaction_cost
                            trades += 1
                        elif sell_surplus > customer_min_surplus and sell_surplus > buy_surplus and inventory < max_inventory:
                            inventory += 1
                            cash -= bid + transaction_cost
                            trades += 1
                fair_price += price_change
            pnl += cash + inventory * fair_price
            run_pnls.append(pnl)
            total_profit += pnl

            if pnl < 0:
                losses += 1
            
            total_trades += trades
        average_trades = total_trades / information_steps
        loss_rate = losses / information_steps
        InformationLossRates.append(loss_rate)
        InformationProfit.append(total_profit / information_steps)
        AvgTrades.append(average_trades)
        PnlStdevs.append(statistics.stdev(run_pnls))
    return InformationProfit, AvgTrades, InformationLossRates, PnlStdevs

def improved_market_maker(Spreads,seed,volatility=.2,trade_steps=1000,spread_steps=100,customer_uncertainty=.8,informed_fraction=.1,informed_trade_probability=.7):
    rng = random.Random(seed)
    SpreadsProfit = []
    AvgTrades = []
    SpreadsLossRate = []
    PnlStdevs = []
    for spread in Spreads: 
        total_profit = 0.0
        total_trades = 0
        losses = 0
        run_pnls = []
        for i in range(spread_steps):
            fair_price = 100.0
            inventory = 0
            cash = 0.0
            max_inventory = 10
            order_flow_bias = .4
            arrival_sensitivity = .7
            trades = 0
            pnl = 0
            transaction_cost = .02
            for t in range(trade_steps):
                price_change = rng.gauss(0, volatility)

                bid = fair_price - spread / 2
                ask = fair_price + spread / 2

                arrival_prob = math.exp(-arrival_sensitivity * spread)
                s = rng.random()
                if s < arrival_prob:
                    if s < informed_fraction:
                        if price_change > 0:
                            if s < informed_trade_probability:
                                buy_probability = (.5 + order_flow_bias)
                            else:
                                buy_probability = (.5 - order_flow_bias)
                        elif price_change <0:
                            if s < informed_trade_probability:
                                buy_probability = (.5 - order_flow_bias)
                            else:
                                buy_probability = (.5 + order_flow_bias)
                        else:
                            buy_probability = .5
                        if s < buy_probability and inventory > -max_inventory:
                            inventory -= 1
                            cash += ask - transaction_cost
                            trades += 1
                        elif s >= buy_probability and inventory < max_inventory:
                            inventory += 1
                            cash -= bid + transaction_cost
                            trades += 1
                    else:
                        customer_min_surplus = .05
                        customer_value = fair_price + rng.gauss(0, customer_uncertainty)
                        buy_surplus = customer_value - ask
                        sell_surplus = bid - customer_value
                        if buy_surplus > customer_min_surplus and buy_surplus >= sell_surplus and inventory > -max_inventory:
                            inventory -= 1
                            cash += ask - transaction_cost
                            trades += 1
                        elif sell_surplus > customer_min_surplus and sell_surplus > buy_surplus and inventory < max_inventory:
                            inventory += 1
                            cash -= bid + transaction_cost
                            trades += 1
                fair_price += price_change
            pnl += cash + inventory * fair_price
            run_pnls.append(pnl)
            total_profit += pnl

            if pnl < 0:
                losses += 1
            
            total_trades += trades
        average_trades = total_trades / spread_steps
        loss_rate = losses / spread_steps
        SpreadsLossRate.append(loss_rate)
        SpreadsProfit.append(total_profit / spread_steps)
        AvgTrades.append(average_trades)
        PnlStdevs.append(statistics.stdev(run_pnls))
    return SpreadsProfit, AvgTrades, SpreadsLossRate, PnlStdevs
def VolOptimalSpread(vol,informed_fraction, seed = 42, customer_uncertainty=.8,informed_trade_probability=.7,spreads = [i/50 for i in range(1,125,1)]):
    SpreadsProfit, AvgTrades, SpreadsLossRate, PnlStdevs = improved_market_maker(spreads, seed=42, volatility=vol,customer_uncertainty=.8,informed_fraction=informed_fraction,informed_trade_probability=.7)
    coefficients_cubic = np.polyfit(spreads, SpreadsProfit, 3)
    a_cubic, b_cubic, c_cubic, d_cubic = coefficients_cubic
    spread_fit_cubic = np.linspace(min(spreads), max(spreads), 100)
    profit_fit_cubic = a_cubic * spread_fit_cubic**3 + b_cubic * spread_fit_cubic**2 + c_cubic * spread_fit_cubic + d_cubic
    optimal_spread = spread_fit_cubic[np.argmax(profit_fit_cubic)]
    return optimal_spread

In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression

In [3]:
volatilities = [.25,.25,.5,.5,.75,.75,1,1]
informs = [.4,.7,.4,.7,.4,.7,.4,.7]
output = []
for i in range(len(volatilities)):
    output.append(VolOptimalSpread(volatilities[i],informs[i]))

df = pd.DataFrame({"volatility": volatilities, "informed_pct": informs, "optimal_spread": output})
df["interaction"] = (df["volatility"] * df["informed_pct"])
X = df[["volatility","informed_pct","interaction"]]
Y = df["optimal_spread"]

model = LinearRegression()
model.fit(X,Y)

def optimal_spread(volatility, informed_pct):
    interaction = volatility * informed_pct

    pred = model.predict([[volatility,informed_pct,interaction]])

    return pred[0]

In [4]:
#Let's see how this model handles our base case
optimal_spread(.2,.1)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


np.float64(1.3961090909090912)

In [5]:
#That is right around the max profitability for that model, lets see how it works when increasing one of the variables

print(optimal_spread(.5,.1))
print(optimal_spread(.2,.5))

1.7270909090909092
1.4583959595959597


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [6]:
def best_avg_profit(volatility,informed_pct,seed = 42):
    SpreadsProfit, AvgTrades, SpreadsLossRate, PnlStdevs = improved_market_maker([optimal_spread(volatility,informed_pct)],seed,volatility = volatility, informed_fraction=informed_pct)
    return SpreadsProfit

In [7]:
best_avg_profit(.2,.1)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[np.float64(68.01652555833496)]